# 01 · Launch Control

Reproduces van der Drift & O'Gorman (2025): **ocean reference → Stage 1 (Ts relaxation) → Stage 2 (fixed Ts)** for each r_v.

How to use:
1. Edit the **LEVERS** cell and run it.
2. Run **Start** once. It submits the ocean run.
3. Run **Advance** whenever you like. It checks every run and takes the next step: resubmit after a walltime kill, extend Stage 1, promote to Stage 2, or mark done. Nothing happens on its own unless `AUTO_ADVANCE = True`.
4. Run **Status** to see the table.

Safety: touch `<experiment>/STOP` (cell at the bottom) to freeze all automatic steps. Every action is logged in `ledger.json`.
Paths, modules and SLURM settings live in `samheat/paths.py`.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ===================== LEVERS =====================
# Every setting of the experiment lives here. Change, then re-run the cell you need below.
# Physics settings are frozen once the experiment starts: to change them, use a new EXPERIMENT name.
EXPERIMENT = "vdd2025_repro"
RV_LIST    = [0, 200, 500, 1000, 2000]     # surface evaporative resistance r_v (s/m); may grow later
TS_OCEAN   = 300.0                          # ocean-reference SST (K)

OCEAN  = dict(days=50, avg_days=10, snd="snd_vdd300K")   # reference sounding = mean of last avg_days
STAGE1 = dict(days=40, extend_days=10, window_days=10, tol_K=0.1, max_days=100,
              tau_sst=311040.0, p_low=400.0, p_upp=600.0)  # Ts relaxation (paper Eq. 2)
STAGE2 = dict(days=60, out_start_day=30, out_every_hr=3, save2D=True, save3D=True)

OVERRIDES = {}          # any other &PARAMETERS key, e.g. {"nrad": 90}. Typos raise an error.
ALLOW_DIURNAL = False   # False = refuse anything that is not perpetual sun like the paper
AUTO_ADVANCE  = False   # True = each job runs advance() for itself when it ends (guarded)
MAX_SUBMITS   = 8       # per run per stage; then the run is marked failed
DRY_RUN       = False   # True = build run folders under <name>_dryrun, submit nothing

CONFIG = dict(name=EXPERIMENT, rv_list=RV_LIST, ts_ocean=TS_OCEAN, ocean=OCEAN, stage1=STAGE1,
              stage2=STAGE2, overrides=OVERRIDES, allow_diurnal=ALLOW_DIURNAL,
              auto_advance=AUTO_ADVANCE, max_submits=MAX_SUBMITS, dry_run=DRY_RUN)

from samheat.paths import SITE
print("experiment folder:", SITE.work_root / EXPERIMENT)


## Start: submit the ocean reference (run once)

In [ ]:
from samheat.stages import Experiment
Experiment(CONFIG).start()

## Advance: take the next step for every run (safe to re-run any time)

In [ ]:
from samheat.stages import Experiment
Experiment(CONFIG).advance()

## Status

In [ ]:
import pandas as pd
from samheat.stages import Experiment
pd.DataFrame(Experiment(CONFIG).status())

## History of one run

In [ ]:
from samheat.stages import Experiment
for line in Experiment(CONFIG).history("rv500"):
    print(line)

## Convergence plot: 600–400 hPa layer T and Ts vs time (ocean run in black, Stage 1 runs in colour)

In [ ]:
import matplotlib.pyplot as plt
from samheat.stages import Experiment
from samheat.io.stat import open_stat
from samheat.convergence import tabs_integral
exp = Experiment(CONFIG)
state = exp.ledger.load()
if not state:
    print("Nothing started yet: run the Start cell first.")
else:
    T_ref = state.get("_meta", {}).get("T_ref")
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    plotted = []
    for key, e in state.items():
        if key == "_meta":
            continue
        d = e.get("stage1_dir") or (e.get("dir") if e.get("stage") in (1, "ocean") or key == "ocean" else None)
        if not d:
            continue
        st = open_stat(d)
        if st is None or st.sizes.get("time", 0) == 0:
            print(f"{key}: no STAT output yet")
            continue
        lt = [tabs_integral(CONFIG["stage1"]["p_low"], CONFIG["stage1"]["p_upp"], st["p"].values, st["TABS"].isel(time=i).values)
              for i in range(st.sizes["time"])]
        style = dict(color="k", lw=2) if key == "ocean" else {}
        a1.plot(st["time"], lt, label=key, **style)
        a2.plot(st["time"], st["SST"], label=key, **style)
        plotted.append(key)
    if T_ref:
        a1.axhline(T_ref, color="k", ls="--", lw=0.8, label="T_ref")
    a1.set_ylabel("600-400 hPa T (K)"); a2.set_ylabel("Ts (K)"); a2.set_xlabel("day")
    if plotted:
        a1.legend(fontsize=8)
        plt.show()
    else:
        plt.close(fig)
        print("No STAT output to plot yet. Stages:", {k: v["stage"] for k, v in state.items() if k != "_meta"})

## Emergency brake

In [ ]:
from samheat.stages import Experiment
exp = Experiment(CONFIG)
(exp.root / "STOP").touch()
print("STOP file created:", exp.root / "STOP", "(delete it to resume)")